# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [15]:
# imports
from openai import OpenAI
from IPython.display import Markdown, display, update_display

In [16]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [17]:
# set up environment
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [18]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [19]:
system_prompt = "You're a helpful tutor that explains in detail and in simple terms. Response format: Markdown"
messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": question}]

In [20]:
def stream_response(stream):
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [21]:
# Get gpt-4o-mini to answer, with streaming
openai = OpenAI()
stream = openai.chat.completions.create(model=MODEL_GPT, messages=messages, stream=True)

stream_response(stream=stream)

Certainly! Let's break down this code line step by step and explain what it does and why.

### Code Breakdown
```python
yield from {book.get("author") for book in books if book.get("author")}
```

### Explanation

1. **Set Comprehension**:
   - The part `{book.get("author") for book in books if book.get("author")}` is a set comprehension.
   - It creates a new set (a collection of unique items) containing authors extracted from the `books` collection.

2. **Iterating Over `books`**:
   - `for book in books`: This loop goes through each item in the `books` iterable. Here, `books` is expected to be a list (or any iterable) of dictionaries.
   - Each `book` is typically a dictionary that represents a book, where keys might include "author", "title", etc.

3. **Extracting the Author**:
   - `book.get("author")`: This tries to retrieve the value associated with the key "author" from each `book` dictionary.
   - The `get()` method is used because it safely returns `None` if the "author" key does not exist, instead of raising an error.

4. **Conditional Filtering**:
   - `if book.get("author")`: This condition checks whether the author exists (i.e., it is not `None` or an empty string). If the author exists, it includes that author's name in the set.

5. **Yielding Values**:
   - `yield from`: This part of the code is used in a generator function. It allows yielding values from an iterable (in this case, the set created).
   - Essentially, it means "yield each item from the set of authors".

### What It Does
- The whole line of code generates a set of unique authors from the `books` iterable while skipping books that do not have an author.
- It yields each unique author's name, one at a time, as part of a generator.

### Why It's Useful
- This code is useful for processing or handling book data where you want to collect unique authors without duplication.
- By using `yield from`, it allows you to incorporate the unique authors into a larger generator function seamlessly, enabling memory-efficient iteration over potentially large datasets.

### Example
Suppose you have the following list of books:

```python
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "author": "Author B"},
    {"title": "Book 3", "author": "Author A"},
    {"title": "Book 4"}  # No author
]
```

If you run the provided line in a generator, it would yield the following unique authors:

1. Author A
2. Author B

You will not get "Author A" twice since we used a set, which only stores unique items.

Feel free to ask if you have more questions about this code or related topics!

In [22]:
# Get Llama 3.2 to answer
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
stream = ollama.chat.completions.create(model=MODEL_LLAMA, messages=messages, stream=True)

stream_response(stream=stream)

**Delegation with `yield from`**
====================================

The given code uses a powerful syntax feature in Python called "delegation" to extract data from multiple sources. Let's break it down:

**Components**

* `{book.get("author") for book in books if book.get("author")}`: This is an expression that generates values.
	+ `for book in books`: Iterates over a collection of objects (`books`).
	+ `if book.get("author")`: Filters the iteration, only including items where the `"author"` key exists and its value is not empty or falsey.
* `yield from`: A syntax extension that allows us to delegate to another iterator.

**How it works**

1. The expression `{book.get("author") for book in books if book.get("author")}` generates an iterator over the desired author names.
2. Instead of iterating over this new iterator, we use `yield from` to "delegate" its iteration to our original loop (`for book in books`). This means the inner loop will iterate over the results generated by the expression outside the `yield from`.
3. The outer loop still iterates over each item in `books`, but for those items that pass the `if condition`, it yields the corresponding author name, provided there exists an `"author"` key.

**Example**
-----------

Let's say we have a list of book objects with varying author information:
```python
books = [
    {"title": "Book A", "author": "John Smith"},
    {"title": "Book B", "author": ""},
    {"title": "Book C"}
]
```

When you run the code, it will yield these values in order: `'John Smith'`, `''` (because `"author"` is missing or falsy), and then stop at the next iteration:

### Code Block
```python
def main():
    books = [
        {"title": "Book A", "author": "John Smith"},
        {"title": "Book B", "author": ""},
        {"title": "Book C"}
    ]

    for author in yield from ({book.get("author") for book in books}):
        print(author)

if __name__ == "__main__":
    main()
```

This will output:
```
John Smith
'' 
```

As you can see, we leveraged the delegating power of `yield from` to extract only the desired authors while managing the inner loop separately.

### Why This is Useful

This particular construct allows for elegant handling of complex data sources where processing a subset of values requires careful consideration. It also promotes good coding habits by promoting "don't repeat yourself" (DRY) principles, minimizing repetition in code.